# 01 · Python Refresher for Fine-Tuning

In plain English, before we ever teach a language model anything, we have to *speak Python well enough to hand it clean data and to read its outputs*. This notebook is a fast, friendly refresher on exactly the Python you'll lean on throughout this course — the data structures, loops, file formats, and a couple of libraries (NumPy and pandas) that show up in every fine-tuning project. No heavy math, no clever tricks. Just the everyday Python a fine-tuner actually uses.

## What you'll learn

- The four core containers: **lists, dicts, tuples, sets** — with machine-learning-flavored examples.
- **Comprehensions and generators**, and why they make data preparation fast and readable.
- **Functions**: default arguments, `*args`/`**kwargs`, and `lambda`.
- **Iteration helpers**: `enumerate`, `zip`, and `range`.
- **Reading and writing files**, especially **JSON** and **JSONL** (the format fine-tuning datasets use).
- **NumPy** basics: arrays, shapes, dtype, vectorized math, broadcasting, `mean`, `argmax`, `reshape`.
- A tiny **pandas** intro: build a small table, filter rows, and count values.
- **Classes**: `__init__`, methods, and `__call__` — and how PyTorch models are "callable" objects.
- **f-strings** and a light touch of **type hints**.

## Why this matters for fine-tuning

Fine-tuning is mostly **data wrangling plus a short training loop**. Roughly 80% of the work is getting your examples into the right shape:

- Your training set is usually a **list of dicts** (each dict is one example), saved as a **JSONL** file.
- You'll **map labels to numbers** with dicts, and **batch** data with loops and comprehensions.
- Model inputs and outputs are **tensors** — multi-dimensional number grids that behave almost exactly like the **NumPy arrays** we practice here.
- A PyTorch model is a **class** you create once and then **call like a function** (`output = model(input)`), which is why `__call__` gets its own section.

Master this notebook and the rest of the course is about *ideas*, not *syntax*.

## Setup

Run the cell below once. The `%pip install` line is **commented out** — uncomment it if you're on Google Colab or a fresh environment that doesn't already have NumPy and pandas.

In [ ]:
# Uncomment the next line on Colab or a fresh environment:
# %pip install numpy pandas

import json          # for reading/writing JSON and JSONL data files
import numpy as np   # NumPy: fast number arrays; "np" is the universal nickname
import pandas as pd  # pandas: tables (DataFrames); "pd" is the universal nickname

print("Imports OK")  # -> Imports OK

## 1. Lists, dicts, tuples, sets

These four built-in containers hold collections of things. You'll use all of them constantly.

- **list** `[...]` — an ordered, changeable sequence. Great for "a bunch of training examples."
- **dict** `{key: value}` — labeled lookups. Great for "label name to label id."
- **tuple** `(...)` — like a list but **cannot be changed** (immutable). Great for fixed pairs like an `(input, label)`.
- **set** `{...}` — an unordered collection of **unique** items. Great for "what labels exist?"

In [ ]:
# A LIST of training examples. Each example is a dict with text + a label.
train = [
    {"text": "I love this product!",      "label": "positive"},
    {"text": "Worst purchase ever.",       "label": "negative"},
    {"text": "It's okay, nothing special", "label": "neutral"},
]
print("number of examples:", len(train))   # -> number of examples: 3
print("first example text:", train[0]["text"])  # index 0 = first item

# A DICT mapping label names to integer ids (models need numbers, not words).
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {i: name for name, i in label2id.items()}  # reverse it (more below)
print("id for 'positive':", label2id["positive"])     # -> 2
print("name for id 0:", id2label[0])                   # -> negative

# A TUPLE: a fixed (text, label_id) pair. You cannot reassign its items.
pair = ("I love this!", 2)
print("tuple pair:", pair, "| text =", pair[0], "| id =", pair[1])

# A SET: the unique labels actually present in the data.
labels_present = {ex["label"] for ex in train}   # set comprehension
print("unique labels:", labels_present)          # order is not guaranteed

**What this does:**
- `train` is a **list of dicts** — the exact shape most fine-tuning datasets take.
- `train[0]["text"]` chains two lookups: get item `0` from the list, then key `"text"` from that dict.
- `label2id["positive"]` looks up a value by key; this is how we turn words into the numbers a model needs.
- `id2label = {i: name for name, i in label2id.items()}` flips keys and values so we can go back from a predicted number to a human-readable label.
- The set `{ex["label"] for ex in train}` automatically removes duplicates, so it tells us which classes exist.

### ✏️ Exercise

Add a fourth training example with label `"positive"` to `train`, then rebuild `labels_present`. How many unique labels are there now?

*Hint:* use `train.append({...})`, then recompute the set comprehension. Adding another `"positive"` should **not** grow the set, because sets only keep unique items.

In [ ]:
# Your turn! Try it, then peek at the sample solution below.
# train.append({"text": "Fantastic!", "label": "positive"})
# labels_present = {ex["label"] for ex in train}
# print(labels_present)   # still 3 unique labels: positive, negative, neutral

**Sample solution:**
```python
train.append({"text": "Fantastic!", "label": "positive"})
labels_present = {ex["label"] for ex in train}
print(len(labels_present))   # -> 3  (duplicates don't count)
```

## 2. Comprehensions and generators

A **comprehension** is a one-line way to build a list, dict, or set by looping. In data prep you'll use them to transform every example at once — for example, turning label words into ids.

A **generator** looks similar but uses `(...)` (or `yield`) and produces items **one at a time, on demand**, instead of building the whole list in memory. That matters when a dataset is too big to fit in RAM all at once.

In [ ]:
texts  = [ex["text"]  for ex in train]                 # list comprehension
ids    = [label2id[ex["label"]] for ex in train]       # map labels -> ids
print("texts:", texts)
print("ids:", ids)

# Dict comprehension: example index -> its label id
index_to_id = {i: label2id[ex["label"]] for i, ex in enumerate(train)}
print("index_to_id:", index_to_id)

# Filtering inside a comprehension: keep only positive examples
positives = [ex["text"] for ex in train if ex["label"] == "positive"]
print("positives:", positives)

# A GENERATOR: note the round brackets. Nothing is computed until we ask.
lengths_gen = (len(t) for t in texts)   # describes work, doesn't do it yet
print("first length on demand:", next(lengths_gen))  # pulls ONE value
print("sum of remaining lengths:", sum(lengths_gen)) # consumes the rest

**What this does:**
- `[label2id[ex["label"]] for ex in train]` loops over every example and converts its label word to a number in a single readable line.
- Adding `if ex["label"] == "positive"` filters as it builds — only matching items are kept.
- `(len(t) for t in texts)` is a **generator**: round brackets mean "lazy." It computes each length only when pulled with `next(...)` or consumed by `sum(...)`. Once consumed, a generator is exhausted and can't be reused.

### ✏️ Exercise

Build a list `short_texts` containing only the texts with fewer than 20 characters.

*Hint:* `len(t) < 20` inside a list comprehension with an `if`.

**Sample solution:**
```python
short_texts = [t for t in texts if len(t) < 20]
print(short_texts)
```

## 3. Functions: defaults, *args / **kwargs, lambdas

Functions package reusable logic. You'll write small helper functions to clean text, build batches, and compute metrics.

- **Default arguments** let a parameter have a fallback value.
- `*args` collects extra **positional** arguments into a tuple.
- `**kwargs` collects extra **keyword** arguments into a dict — you'll see this a lot in library configs.
- A **lambda** is a tiny one-line anonymous function, handy for `sorted(..., key=...)`.

In [ ]:
def normalize(text, lowercase=True):
    """Clean a piece of text. 'lowercase' has a DEFAULT value of True."""
    text = text.strip()                 # remove leading/trailing whitespace
    if lowercase:                       # only lowercase if asked (default: yes)
        text = text.lower()
    return text

print(normalize("  Hello WORLD  "))            # -> hello world
print(normalize("  Hello WORLD  ", lowercase=False))  # -> Hello WORLD

def describe_run(*args, **kwargs):
    """*args = extra positional values; **kwargs = extra named values."""
    print("positional args (a tuple):", args)
    print("keyword args (a dict):", kwargs)

describe_run("bert", "sst2", epochs=3, lr=2e-5)
# -> positional args (a tuple): ('bert', 'sst2')
# -> keyword args (a dict): {'epochs': 3, 'lr': 2e-05}

# A lambda: sort the examples by text length (shortest first).
by_length = sorted(train, key=lambda ex: len(ex["text"]))
print("shortest example:", by_length[0]["text"])

**What this does:**
- `def normalize(text, lowercase=True)` — calling `normalize(s)` uses the default; passing `lowercase=False` overrides it.
- `*args` packs leftover positional arguments into a **tuple**; `**kwargs` packs leftover keyword arguments into a **dict**. Libraries use `**kwargs` so you can pass many optional settings.
- `lambda ex: len(ex["text"])` is a throwaway function that tells `sorted` what to sort by — here, text length.

### ✏️ Exercise

Write a function `to_id(label, mapping=label2id)` that returns the integer id for a label. Test it with `to_id("neutral")`.

*Hint:* just `return mapping[label]`. (We'll see a gotcha about mutable defaults in the "Common mistakes" section — using a dict only for lookup like this is safe.)

**Sample solution:**
```python
def to_id(label, mapping=label2id):
    return mapping[label]

print(to_id("neutral"))   # -> 1
```

## 4. Iteration helpers: `enumerate`, `zip`, `range`

These three make loops clean and bug-free.

- `range(n)` produces `0, 1, ..., n-1` — for counting.
- `enumerate(seq)` gives you `(index, item)` pairs — no manual counter needed.
- `zip(a, b)` walks two (or more) sequences **in lockstep**, pairing them up.

In [ ]:
# range: classic counting loop
for i in range(3):
    print("step", i)        # -> step 0 / step 1 / step 2

# enumerate: get index AND item together
for i, ex in enumerate(train):
    print(f"example {i}: {ex['label']}")   # f-string formatting (more later)

# zip: pair texts with their ids, position by position
for text, the_id in zip(texts, ids):
    print(the_id, "<-", text)

# zip is also a neat way to "unzip" pairs back into two lists
pairs = list(zip(texts, ids))
back_texts, back_ids = zip(*pairs)   # the * unpacks the list of pairs
print("unzipped ids:", back_ids)

**What this does:**
- `enumerate(train)` saves you from writing `i = 0; i += 1` by hand and prevents off-by-one bugs.
- `zip(texts, ids)` stops at the **shortest** input, so mismatched lengths silently truncate — keep paired lists the same length.
- `zip(*pairs)` uses `*` to unpack a list of `(text, id)` tuples back into separate `texts` and `ids` groups.

### ✏️ Exercise

Use `enumerate` to print only the **even-indexed** examples (indices 0, 2, ...).

*Hint:* `if i % 2 == 0:` inside the loop. `%` is the remainder operator.

**Sample solution:**
```python
for i, ex in enumerate(train):
    if i % 2 == 0:
        print(i, ex["text"])
```

## 5. Files, JSON, and JSONL (the fine-tuning data format)

**JSON** (JavaScript Object Notation) is a text format for structured data — Python dicts and lists map onto it directly.

**JSONL** ("JSON Lines") means **one complete JSON object per line**. This is the format almost every fine-tuning toolkit expects for datasets, because you can stream it line-by-line without loading the whole file:

```
{"text": "I love this!", "label": "positive"}
{"text": "Awful.",        "label": "negative"}
```

Preview: later in this course you'll save your training data as a `.jsonl` file and point the trainer at it.

In [ ]:
# --- Write a JSONL file: one JSON object per line ---
with open("demo_train.jsonl", "w", encoding="utf-8") as f:   # 'w' = write mode
    for ex in train:
        line = json.dumps(ex)        # turn the dict into a JSON string
        f.write(line + "\n")         # one object per line -> the "L" in JSONL
print("wrote demo_train.jsonl")

# --- Read it back, line by line ---
loaded = []
with open("demo_train.jsonl", "r", encoding="utf-8") as f:   # 'r' = read mode
    for line in f:
        loaded.append(json.loads(line))   # parse each line back into a dict
print("loaded", len(loaded), "examples")  # -> loaded 3 examples
print(loaded[0])                           # -> {'text': 'I love this!', ...}

# --- For comparison: a single regular JSON file (whole list at once) ---
with open("demo_train.json", "w", encoding="utf-8") as f:
    json.dump(train, f, indent=2)   # indent=2 makes it human-readable
print("wrote demo_train.json (one big array)")

**What this does:**
- `with open(...) as f:` opens a file and **automatically closes** it when the block ends — always use this pattern.
- `json.dumps(ex)` turns one dict into a JSON **string**; `f.write(line + "\n")` adds the newline that separates JSONL records.
- `json.loads(line)` parses a JSON string back into a Python dict. (Tip: `dumps`/`loads` work on strings; `dump`/`load` work directly on files.)
- The `.json` version stores the **entire list** as one big array — fine for small data, but JSONL is preferred for training because it streams.

### ✏️ Exercise

Read `demo_train.jsonl` and count how many examples have `label == "positive"`.

*Hint:* loop over `loaded`, or use a comprehension with `sum(1 for ex in loaded if ...)`.

**Sample solution:**
```python
n_pos = sum(1 for ex in loaded if ex["label"] == "positive")
print(n_pos)   # -> 1 (with the original 3 examples)
```

## 6. NumPy: arrays, shapes, dtype, vectorized math

**NumPy** gives us the **array** — a grid of numbers that supports fast math on the whole grid at once (no slow Python loops). This is crucial because **tensors** (the inputs and outputs of neural networks) behave almost exactly like NumPy arrays: same idea of *shape*, *dtype*, vectorized math, and broadcasting.

Key words:
- **shape** — the size along each dimension, e.g. `(2, 3)` means 2 rows and 3 columns.
- **dtype** — the data type of the numbers, e.g. `int64` or `float32`.

In [ ]:
a = np.array([1, 2, 3, 4])          # a 1-D array (a vector)
print("array:", a)
print("shape:", a.shape)            # -> (4,)
print("dtype:", a.dtype)            # -> int64 (whole numbers)

# Vectorized math: operations apply to EVERY element, no loop needed.
print("a * 2:", a * 2)              # -> [2 4 6 8]
print("a + 10:", a + 10)            # -> [11 12 13 14]

m = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])     # a 2-D array (a matrix)
print("matrix shape:", m.shape)     # -> (2, 3)
print("matrix dtype:", m.dtype)     # -> float64 (note the decimals)

print("overall mean:", m.mean())            # -> 3.5
print("mean per column:", m.mean(axis=0))   # average down each column
print("mean per row:", m.mean(axis=1))      # average across each row

**What this does:**
- `np.array([...])` builds an array; `.shape` and `.dtype` describe it (the two things you'll check most when debugging models).
- `a * 2` multiplies **every** element at once — this "vectorized" style is both faster and clearer than a Python loop.
- `axis=0` collapses **rows** (giving a per-column result); `axis=1` collapses **columns** (per-row). Remembering which axis is which is a common source of bugs.

In [ ]:
# argmax: index of the largest value — how we read a model's prediction.
scores = np.array([0.1, 0.7, 0.2])   # imagine these are class probabilities
predicted_id = scores.argmax()        # index of the biggest score
print("predicted id:", predicted_id)         # -> 1
print("predicted label:", id2label[int(predicted_id)])  # -> neutral

# reshape: rearrange the same numbers into a new shape (total count must match).
flat = np.arange(6)                  # -> [0 1 2 3 4 5], shape (6,)
grid = flat.reshape(2, 3)            # -> 2 rows x 3 cols
print("reshaped to 2x3:\n", grid)

# Broadcasting: a small array is "stretched" to match a bigger one.
row = np.array([10, 20, 30])         # shape (3,)
print("grid + row (broadcast):\n", grid + row)   # row added to EACH row

**What this does:**
- `scores.argmax()` returns the **position** of the highest value. Models output a score per class; `argmax` turns those scores into the chosen class id, which we map back to a label.
- `flat.reshape(2, 3)` keeps the same 6 numbers but rearranges them; the new shape's total size must equal the old one (`2*3 == 6`).
- **Broadcasting**: `grid + row` works because the `(3,)` row is automatically applied to every row of the `(2, 3)` grid — no manual looping.

### ✏️ Exercise

Make a NumPy array `probs = np.array([0.2, 0.5, 0.3])`, find the predicted id with `argmax`, and print its label using `id2label`.

*Hint:* wrap the argmax result in `int(...)` before using it as a dict key.

**Sample solution:**
```python
probs = np.array([0.2, 0.5, 0.3])
pid = int(probs.argmax())   # -> 1
print(id2label[pid])        # -> neutral
```

## 7. A tiny pandas intro

**pandas** gives us the **DataFrame** — a table with named columns, like a spreadsheet in Python. You'll use it to inspect, clean, and filter datasets before fine-tuning. Here we build a small table of "leads."

In [ ]:
# Build a DataFrame from a dict of columns (each key is a column name).
leads = pd.DataFrame({
    "family_size": [4, 1, 3, 2, 5],
    "income":      [55000, 30000, 72000, 48000, 90000],
    "rent_or_own": ["rent", "rent", "own", "rent", "own"],
})
print(leads)            # prints the whole table with an index column
print("\nshape (rows, cols):", leads.shape)   # -> (5, 3)

# Select a single column (returns a Series, like a labeled 1-D array).
print("\nincomes:\n", leads["income"])

# Filter rows with a boolean condition (a "mask").
homeowners = leads[leads["rent_or_own"] == "own"]
print("\nhomeowners:\n", homeowners)

# Combine conditions with & (and) / | (or) — wrap each condition in ().
big_earners = leads[(leads["income"] > 50000) & (leads["family_size"] >= 3)]
print("\nbig earners with big families:\n", big_earners)

# value_counts: how many of each category? Super handy for checking labels.
print("\nrent vs own counts:\n", leads["rent_or_own"].value_counts())

**What this does:**
- `pd.DataFrame({...})` builds a table where each dict key becomes a **column**.
- `leads["income"]` selects one column; `leads[mask]` keeps only rows where the boolean `mask` is `True`.
- In pandas you must use `&`/`|` (not `and`/`or`) to combine conditions, and **each condition needs its own parentheses** — a classic beginner trap.
- `value_counts()` tallies how often each value appears — exactly how you'd check whether your training labels are balanced.

### ✏️ Exercise

From `leads`, select only the renters (`rent_or_own == "rent"`) and print how many there are.

*Hint:* build a mask, then check `.shape[0]` (number of rows) or use `len(...)`.

**Sample solution:**
```python
renters = leads[leads["rent_or_own"] == "rent"]
print(len(renters))   # -> 3
```

## 8. Classes: `__init__`, methods, and `__call__`

A **class** is a blueprint for an object that bundles **data** (attributes) with **behavior** (methods).

- `__init__` is the **constructor** — it runs when you create the object and sets up its data.
- A **method** is a function defined inside the class; the first parameter is always `self` (the object itself).
- `__call__` is special: if a class defines it, you can use an instance **like a function** — `obj(x)` runs `obj.__call__(x)`.

Preview: this is exactly how **PyTorch models** work. You build the model once (`model = MyModel(...)`), then *call* it on inputs (`outputs = model(inputs)`), which secretly runs its `__call__` / `forward` logic.

In [ ]:
class LabelEncoder:
    """A tiny stand-in for the kind of callable object PyTorch uses."""

    def __init__(self, label2id):       # constructor: runs at creation time
        self.label2id = label2id        # store the mapping as an attribute
        self.id2label = {i: name for name, i in label2id.items()}

    def encode(self, label):            # a normal method (note 'self')
        return self.label2id[label]     # word -> id

    def decode(self, idx):              # another method
        return self.id2label[idx]       # id -> word

    def __call__(self, label):          # makes the OBJECT callable like a func
        return self.encode(label)       # so encoder("positive") just works

# Create one instance (this triggers __init__).
encoder = LabelEncoder(label2id)

print("method call  encoder.encode('positive'):", encoder.encode("positive"))  # -> 2
print("callable call encoder('positive'):", encoder("positive"))               # -> 2
print("decode 0 back to a word:", encoder.decode(0))                           # -> negative

**What this does:**
- `__init__(self, label2id)` saves the mapping onto the object via `self.label2id`, so every method can reach it.
- `encoder.encode("positive")` calls a method explicitly; `encoder("positive")` calls the object **directly** because we defined `__call__`. Both return the same thing here.
- That "call the object like a function" pattern is precisely why, in PyTorch, you write `model(inputs)` instead of `model.forward(inputs)` — calling the object is the recommended way.

### ✏️ Exercise

Add a method `labels(self)` to `LabelEncoder` that returns a list of all label names it knows. Create a new encoder and call it.

*Hint:* `return list(self.label2id.keys())`.

**Sample solution:**
```python
class LabelEncoder2(LabelEncoder):
    def labels(self):
        return list(self.label2id.keys())

enc = LabelEncoder2(label2id)
print(enc.labels())   # -> ['negative', 'neutral', 'positive']
```

## 9. f-strings and a light touch of type hints

**f-strings** let you drop variables straight into text with `f"...{value}..."`. They're the easiest way to build readable log messages during training.

**Type hints** are optional notes about what types a function expects and returns, like `def f(x: int) -> str:`. Python doesn't enforce them, but they make code easier to read and let editors catch mistakes — and you'll see them all over real ML codebases.

In [ ]:
epoch = 3
loss = 0.04719
acc = 0.912

# f-strings: {var} drops the value in; :.3f means 3 decimal places, :.1% a percent.
print(f"epoch {epoch}: loss={loss:.3f}, accuracy={acc:.1%}")
# -> epoch 3: loss=0.047, accuracy=91.2%

# A function with type hints: takes a list of dicts, returns a float.
def average_text_length(examples: list[dict]) -> float:
    """Hints say: input is a list of dicts, output is a float."""
    lengths = [len(ex["text"]) for ex in examples]
    return sum(lengths) / len(lengths)   # returns a float

avg = average_text_length(train)
print(f"average text length: {avg:.1f} characters")

**What this does:**
- `f"...{loss:.3f}..."` formats the number to 3 decimals; `{acc:.1%}` shows it as a percentage with one decimal. Format specs keep logs tidy.
- `examples: list[dict]` and `-> float` are **type hints**: documentation that tools can check. They do **not** change how the code runs — passing the wrong type won't crash on the hint alone.

### ✏️ Exercise

Write a one-line f-string that prints `"Model has 3 labels: negative, neutral, positive"` using `label2id`.

*Hint:* `len(label2id)` for the count and `", ".join(label2id)` to list the keys.

**Sample solution:**
```python
print(f"Model has {len(label2id)} labels: {', '.join(label2id)}")
# -> Model has 3 labels: negative, neutral, positive
```

## Common mistakes & how to debug them

These four bite almost every beginner. Recognize the symptom, apply the fix.

| Mistake | What goes wrong | Fix |
|---|---|---|
| **Mutable default argument** | `def f(x, acc=[])` — the same list is *reused* across calls, so it secretly grows between calls. | Use `None` as the default, then create the list inside: `def f(x, acc=None): acc = [] if acc is None else acc`. |
| **Integer vs float surprise** | In older code `1/2` could give `0`; mixing int arrays in NumPy can truncate or overflow. Dividing two int arrays may not give what you expect. | Make numbers floats when you mean fractions: `1/2` is fine in Python 3, but for NumPy use `np.array([...], dtype=float)` or multiply by `1.0`. |
| **Modifying a list while iterating** | Removing items from a list inside a `for` loop over that same list skips elements or crashes. | Iterate over a **copy** (`for x in items[:]:`) or, better, build a new list with a comprehension. |
| **Shape mismatch in NumPy** | `a + b` raises `ValueError: operands could not be broadcast together` when shapes don't line up. | Print `a.shape` and `b.shape` first. Use `reshape` so dimensions align, and recall broadcasting rules. |

Below: the mutable-default and modify-while-iterating bugs shown live, with their fixes.

In [ ]:
# --- BUG 1: mutable default argument ---
def add_item_bad(x, bucket=[]):     # DON'T: the list is created ONCE, shared
    bucket.append(x)
    return bucket

print(add_item_bad(1))   # -> [1]
print(add_item_bad(2))   # -> [1, 2]  (surprise! the old 1 is still there)

# FIX: default to None, create a fresh list inside.
def add_item_good(x, bucket=None):
    bucket = [] if bucket is None else bucket
    bucket.append(x)
    return bucket

print(add_item_good(1))  # -> [1]
print(add_item_good(2))  # -> [2]  (fresh each time)

In [ ]:
# --- BUG 3: modifying a list while iterating ---
nums = [1, 2, 3, 4, 5, 6]

# WRONG: removing during iteration skips elements.
# for n in nums:
#     if n % 2 == 0:
#         nums.remove(n)   # this misbehaves!

# RIGHT: build a new list with a comprehension.
odds = [n for n in nums if n % 2 != 0]
print(odds)   # -> [1, 3, 5]

# --- BUG 4: shape mismatch — always check shapes first ---
a = np.ones((2, 3))   # shape (2, 3)
b = np.array([1, 2, 3])  # shape (3,)  -> broadcasts fine
print((a + b).shape)  # -> (2, 3)
# c = np.array([1, 2]) # shape (2,) would FAIL: print a.shape and c.shape to debug

## Summary

- **Containers:** lists hold ordered examples, dicts map labels to ids, tuples are fixed pairs, sets hold unique values.
- **Comprehensions** transform data in one readable line; **generators** stream items lazily for big datasets.
- **Functions** support defaults, `*args`/`**kwargs`, and `lambda` for quick keys.
- **`enumerate`, `zip`, `range`** keep loops clean and bug-free.
- **JSON/JSONL:** fine-tuning datasets are JSONL — one JSON object per line — read and written with the `json` module.
- **NumPy** arrays have a *shape* and *dtype*, support vectorized math and broadcasting, and behave like the tensors models use; `mean`, `argmax`, and `reshape` are everyday tools.
- **pandas** DataFrames let you inspect, filter, and `value_counts()` your data before training.
- **Classes** bundle data + behavior; `__call__` makes an object callable, which is exactly how PyTorch models are used (`model(inputs)`).
- **f-strings** make clean logs; **type hints** document intent.
- Watch out for mutable defaults, int/float surprises, modifying lists while iterating, and NumPy shape mismatches.

## What to learn next

Next up: **`02_ml_fundamentals.ipynb`**, where we turn this Python into a mental model of *machine learning itself* — features, labels, training vs. evaluation, loss, and what "learning" actually means. Everything you practiced here (arrays, dicts of examples, the `model(inputs)` pattern) will reappear with meaning.

See you in notebook 02!